In [8]:
!pip -q install transformers==4.40.0 sentence-transformers==2.7.0 torch==2.2.0 PyPDF2==3.0.1 pdfplumber==0.11.0 langchain==0.1.20 langchain-community==0.0.38 faiss-cpu==1.8.0 chromadb==0.5.0 groq==0.5.0 together==1.2.1 ollama==0.1.9 rouge-score==0.1.2 sacrebleu==2.4.0 nltk==3.8.1 pandas==2.2.0 numpy==1.26.4 tqdm==4.66.2 python-dotenv==1.0.1 matplotlib==3.8.4 seaborn==0.13.2 tabulate==0.9.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!pip -q install --force-reinstall --no-cache-dir "numpy==1.26.4"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 220.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.0 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is

In [2]:
import numpy as np
print("NumPy:", np.__version__)  # should be 1.26.4

import pdfplumber, PyPDF2, faiss, chromadb, pandas, tqdm
from sentence_transformers import SentenceTransformer
print("Core imports OK")


NumPy: 1.26.4
Core imports OK


In [5]:
import os
for d in ["data/raw", "data/chunks", "data/embeddings", "vectordb", "logs", "outputs"]:
    os.makedirs(d, exist_ok=True)
print("Folders created in current Colab runtime.")
!pwd
!find . -maxdepth 2 -type d | sort


Folders created in current Colab runtime.
/content
.
./.config
./.config/configurations
./.config/logs
./data
./data/chunks
./data/embeddings
./data/raw
./logs
./outputs
./sample_data
./vectordb
./.venv
./.venv/bin
./.venv/include
./.venv/lib


In [7]:
from google.colab import files
import shutil, os

uploaded = files.upload()
for fn in uploaded.keys():
    shutil.move(fn, f"data/raw/{fn}")

!ls -lh data/raw


Saving 2_2021_11_16!06_21_15_AM.pdf to 2_2021_11_16!06_21_15_AM.pdf
Saving 3-MEDICAL-IMAGING-TECHNIQUES.pdf to 3-MEDICAL-IMAGING-TECHNIQUES.pdf
Saving CBSE_MH_Manual.pdf to CBSE_MH_Manual.pdf
Saving diabetes.pdf to diabetes.pdf
Saving Immunization.pdf to Immunization.pdf
Saving infectious-diseases-2022-virtual-abstract-book.pdf to infectious-diseases-2022-virtual-abstract-book.pdf
Saving introduction-to-nutrition.pdf to introduction-to-nutrition.pdf
Saving introduction-to-public-health.pdf to introduction-to-public-health.pdf
Saving RAND_WR467.pdf to RAND_WR467.pdf
Saving what-is-cancer.pdf to what-is-cancer.pdf
total 38M
-rw-r--r-- 1 root root 420K Mar 31 15:40 '2_2021_11_16!06_21_15_AM.pdf'
-rw-r--r-- 1 root root 8.8M Mar 31 15:40  3-MEDICAL-IMAGING-TECHNIQUES.pdf
-rw-r--r-- 1 root root  18M Mar 31 15:40  CBSE_MH_Manual.pdf
-rw-r--r-- 1 root root 424K Mar 31 15:40  diabetes.pdf
-rw-r--r-- 1 root root 4.2M Mar 31 15:40  Immunization.pdf
-rw-r--r-- 1 root root 1.4M Mar 31 15:40  infect

In [8]:
"""
PHASE 1 — Document Ingestion & Preprocessing (Healthcare)
=========================================================
What this does:
- Loads PDFs and text files from data/raw/
- Cleans noise (headers, footers, page numbers, weird chars)
- Saves clean text per document to data/raw/cleaned/
- Saves ingestion summary to data/raw/ingestion_summary.json
"""

import os
import re
import json
import pdfplumber
import PyPDF2
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

RAW_DIR = Path(os.getenv("RAW_DATA_DIR", "data/raw"))
CLEANED_DIR = RAW_DIR / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)


def clean_text(text: str) -> str:
    """Remove common PDF noise."""
    text = re.sub(r'[-–]\s*\d+\s*[-–]', '', text)
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)

    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if len(stripped) > 3 and re.search(r'[a-zA-Z]', stripped):
            cleaned_lines.append(line)
        elif stripped == '':
            cleaned_lines.append('')

    return '\n'.join(cleaned_lines).strip()


def load_pdf_pdfplumber(filepath: Path) -> str:
    """Primary PDF loader."""
    text = ""
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"  pdfplumber failed for {filepath.name}: {e}. Trying PyPDF2...")
        text = load_pdf_pypdf2(filepath)
    return text


def load_pdf_pypdf2(filepath: Path) -> str:
    """Fallback PDF loader."""
    text = ""
    try:
        with open(filepath, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"  PyPDF2 also failed for {filepath.name}: {e}")
    return text


def load_txt(filepath: Path) -> str:
    """Load plain text/markdown file."""
    try:
        return filepath.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"  Failed to load {filepath.name}: {e}")
        return ""


def ingest_documents() -> list[dict]:
    """Load + clean all docs from RAW_DIR."""
    supported_extensions = {".pdf", ".txt", ".md"}
    files = [f for f in RAW_DIR.iterdir() if f.suffix.lower() in supported_extensions]

    if not files:
        print(f"\n[!] No documents found in {RAW_DIR}/")
        print("    Upload your healthcare PDFs/TXT/MD files to data/raw and re-run.\n")
        return []

    print(f"\n[+] Found {len(files)} document(s) in {RAW_DIR}/\n")
    documents = []

    for filepath in tqdm(files, desc="Loading documents"):
        print(f"\n  Processing: {filepath.name}")

        if filepath.suffix.lower() == ".pdf":
            raw_text = load_pdf_pdfplumber(filepath)
            source_type = "pdf"
        else:
            raw_text = load_txt(filepath)
            source_type = "text"

        if not raw_text.strip():
            print(f"  [!] Empty document, skipping: {filepath.name}")
            continue

        clean = clean_text(raw_text)

        doc = {
            "filename": filepath.name,
            "source_type": source_type,
            "raw_char_count": len(raw_text),
            "clean_char_count": len(clean),
            "clean_text": clean,
        }
        documents.append(doc)

        clean_path = CLEANED_DIR / f"{filepath.stem}_clean.txt"
        clean_path.write_text(clean, encoding="utf-8")
        print(f"  Saved clean text: {clean_path.name} ({len(clean):,} chars)")

    return documents


def save_ingestion_summary(documents: list[dict]):
    """Save ingestion stats."""
    summary = []
    for doc in documents:
        summary.append({
            "filename": doc["filename"],
            "source_type": doc["source_type"],
            "raw_chars": doc["raw_char_count"],
            "clean_chars": doc["clean_char_count"],
            "compression_ratio": round(doc["clean_char_count"] / max(doc["raw_char_count"], 1), 3),
        })

    summary_path = RAW_DIR / "ingestion_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"\n[+] Ingestion summary saved to {summary_path}")
    return summary


In [10]:
documents = ingest_documents()
if documents:
    save_ingestion_summary(documents)



[+] Found 10 document(s) in data/raw/



Loading documents:   0%|          | 0/10 [00:00<?, ?it/s]


  Processing: CBSE_MH_Manual.pdf


Loading documents:  10%|█         | 1/10 [00:09<01:24,  9.33s/it]

  Saved clean text: CBSE_MH_Manual_clean.txt (136,183 chars)

  Processing: 3-MEDICAL-IMAGING-TECHNIQUES.pdf


...

  Saved clean text: Immunization_clean.txt (31,411 chars)

  Processing: 2_2021_11_16!06_21_15_AM.pdf


Loading documents:  90%|█████████ | 9/10 [00:35<00:02,  2.93s/it]

  Saved clean text: 2_2021_11_16!06_21_15_AM_clean.txt (16,170 chars)

  Processing: RAND_WR467.pdf


Loading documents: 100%|██████████| 10/10 [00:39<00:00,  3.92s/it]

  Saved clean text: RAND_WR467_clean.txt (62,984 chars)

[+] Ingestion summary saved to data/raw/ingestion_summary.json


In [11]:
"""
PHASE 2 — Text Chunking
========================
What this does:
- Reads cleaned documents from data/raw/cleaned/
- Splits each document into chunks using 3 different chunk sizes: 256, 512, 1024
- Uses overlapping windows so context doesn't get cut at boundaries
- Saves all chunks to data/chunks/ as JSON
- This is where your first experiment begins: chunk size comparison

Why chunking matters:
- Too small (256): misses context, answers feel incomplete
- Too large (1024): irrelevant content bleeds into retrieved chunks
- 512 is usually the sweet spot but you MUST show this experimentally

How to use:
    python phase2_chunking.py
"""

import os
import json
from pathlib import Path
from typing import List
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

CLEANED_DIR = Path(os.getenv("RAW_DATA_DIR", "data/raw")) / "cleaned"
CHUNKS_DIR = Path(os.getenv("CHUNKS_DIR", "data/chunks"))
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# Chunk sizes to experiment with — these are in CHARACTERS not tokens
# approx: 1 token ≈ 4 chars, so 256 tokens ≈ 1024 chars
CHUNK_CONFIGS = [
    {"size": 256,  "overlap": 50,  "label": "small"},
    {"size": 512,  "overlap": 50,  "label": "medium"},
    {"size": 1024, "overlap": 50,  "label": "large"},
]

# Character multiplier (token to char conversion approximation)
TOKEN_TO_CHAR = 4


# ─── Chunking Logic ───────────────────────────────────────────────────────────

def chunk_text_sliding_window(
    text: str,
    chunk_size_tokens: int,
    overlap_tokens: int,
    filename: str
) -> List[dict]:
    """
    Sliding window chunker.
    Converts token counts to character counts for simplicity (no tokenizer needed).
    Each chunk carries metadata so you know exactly where it came from.
    """
    chunk_size_chars = chunk_size_tokens * TOKEN_TO_CHAR
    overlap_chars    = overlap_tokens * TOKEN_TO_CHAR
    step             = chunk_size_chars - overlap_chars

    chunks = []
    start  = 0
    idx    = 0

    while start < len(text):
        end = min(start + chunk_size_chars, len(text))
        chunk_text = text[start:end].strip()

        if len(chunk_text) > 50:  # skip tiny fragments
            chunks.append({
                "chunk_id":     f"{filename}_chunk_{idx}",
                "source":       filename,
                "chunk_index":  idx,
                "start_char":   start,
                "end_char":     end,
                "chunk_size":   chunk_size_tokens,
                "text":         chunk_text,
                "char_length":  len(chunk_text)
            })
            idx += 1

        if end == len(text):
            break
        start += step

    return chunks


def chunk_by_paragraph(text: str, filename: str, max_chunk_tokens: int = 512) -> List[dict]:
    """
    Alternative chunker: respects paragraph boundaries.
    Better for legal documents where each paragraph is a logical unit.
    Paragraphs that are too long get split further.
    """
    max_chars = max_chunk_tokens * TOKEN_TO_CHAR
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    chunks = []
    current_chunk = ""
    idx = 0

    for para in paragraphs:
        # If adding this paragraph exceeds max size, save current and start new
        if len(current_chunk) + len(para) > max_chars and current_chunk:
            chunks.append({
                "chunk_id":    f"{filename}_para_chunk_{idx}",
                "source":      filename,
                "chunk_index": idx,
                "chunk_size":  max_chunk_tokens,
                "text":        current_chunk.strip(),
                "char_length": len(current_chunk.strip()),
                "strategy":    "paragraph"
            })
            idx += 1
            current_chunk = para
        else:
            current_chunk += "\n\n" + para if current_chunk else para

    # Don't forget the last chunk
    if current_chunk.strip():
        chunks.append({
            "chunk_id":    f"{filename}_para_chunk_{idx}",
            "source":      filename,
            "chunk_index": idx,
            "chunk_size":  max_chunk_tokens,
            "text":        current_chunk.strip(),
            "char_length": len(current_chunk.strip()),
            "strategy":    "paragraph"
        })

    return chunks


# ─── Main Chunking Loop ───────────────────────────────────────────────────────

def run_chunking():
    """
    For each cleaned document, apply all chunk size configurations.
    Saves separate JSON file per (document, chunk_size) combo.
    """
    clean_files = list(CLEANED_DIR.glob("*_clean.txt"))

    if not clean_files:
        print(f"\n[!] No cleaned files found in {CLEANED_DIR}/")
        print("    Run phase1_ingestion.py first.\n")
        return

    print(f"\n[+] Found {len(clean_files)} cleaned document(s)")
    print(f"[+] Running {len(CHUNK_CONFIGS)} chunk size experiments\n")

    all_stats = []

    for filepath in tqdm(clean_files, desc="Documents"):
        text = filepath.read_text(encoding='utf-8')
        doc_name = filepath.stem.replace('_clean', '')

        print(f"\n  Document: {filepath.name} ({len(text):,} chars)")

        for config in CHUNK_CONFIGS:
            size    = config["size"]
            overlap = config["overlap"]
            label   = config["label"]

            # Sliding window chunks
            chunks = chunk_text_sliding_window(text, size, overlap, doc_name)

            # Save to JSON
            output_path = CHUNKS_DIR / f"{doc_name}_chunks_{size}.json"
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(chunks, f, indent=2, ensure_ascii=False)

            stat = {
                "document":    doc_name,
                "chunk_size":  size,
                "label":       label,
                "num_chunks":  len(chunks),
                "avg_chars":   round(sum(c['char_length'] for c in chunks) / max(len(chunks), 1)),
                "output_file": output_path.name
            }
            all_stats.append(stat)

            print(f"    chunk_size={size:>4} | {len(chunks):>4} chunks | avg {stat['avg_chars']:>5} chars → {output_path.name}")

        # Also do paragraph-based chunking as a bonus strategy
        para_chunks = chunk_by_paragraph(text, doc_name)
        para_path = CHUNKS_DIR / f"{doc_name}_chunks_paragraph.json"
        with open(para_path, 'w', encoding='utf-8') as f:
            json.dump(para_chunks, f, indent=2, ensure_ascii=False)
        print(f"    chunk_size=paragraph | {len(para_chunks):>4} chunks → {para_path.name}")

    # Save stats summary
    stats_path = CHUNKS_DIR / "chunking_stats.json"
    with open(stats_path, 'w') as f:
        json.dump(all_stats, f, indent=2)

    print("\n─── Chunking Summary ────────────────────────────────────")
    print(f"  {'Document':<30} {'Size':>6} {'Chunks':>8} {'Avg chars':>10}")
    print("  " + "-" * 58)
    for s in all_stats:
        print(f"  {s['document']:<30} {s['chunk_size']:>6} {s['num_chunks']:>8} {s['avg_chars']:>10}")

    print(f"\n  Stats saved to {stats_path}")
    print("\n[✓] Phase 2 complete. Run phase3_embeddings.py next.")


# ─── Entry Point ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("  PHASE 2 — Text Chunking (3 chunk sizes + paragraph)")
    print("=" * 60)
    run_chunking()


  PHASE 2 — Text Chunking (3 chunk sizes + paragraph)

[+] Found 10 cleaned document(s)
[+] Running 3 chunk size experiments



Documents: 100%|██████████| 10/10 [00:00<00:00, 167.08it/s]


  Document: introduction-to-nutrition_clean.txt (29,187 chars)
    chunk_size= 256 |   36 chunks | avg  1005 chars → introduction-to-nutrition_chunks_256.json
    chunk_size= 512 |   16 chunks | avg  2011 chars → introduction-to-nutrition_chunks_512.json
    chunk_size=1024 |    8 chunks | avg  3823 chars → introduction-to-nutrition_chunks_1024.json
    chunk_size=paragraph |    4 chunks → introduction-to-nutrition_chunks_paragraph.json

  Document: what-is-cancer_clean.txt (5,550 chars)
    chunk_size= 256 |    7 chunks | avg   964 chars → what-is-cancer_chunks_256.json
    chunk_size= 512 |    3 chunks | avg  1983 chars → what-is-cancer_chunks_512.json
    chunk_size=1024 |    2 chunks | avg  2875 chars → what-is-cancer_chunks_1024.json
    chunk_size=paragraph |    1 chunks → what-is-cancer_chunks_paragraph.json

  Document: Immunization_clean.txt (31,411 chars)
    chunk_size= 256 |   38 chunks | avg  1021 chars → Immunization_chunks_256.json
    chunk_size= 512 |   17 chunks | av

In [13]:
"""
PHASE 3 — Embedding Generation
================================
What this does:
- Loads chunks from data/chunks/
- Embeds every chunk using TWO different embedding models
- Model 1: all-MiniLM-L6-v2     → 384 dimensions, fast, lightweight
- Model 2: BAAI/bge-base-en-v1.5 → 768 dimensions, better semantic understanding
- Saves embeddings as numpy arrays + metadata JSON
- This is your second experiment: embedding model comparison

Why two models matter:
- Different models capture different aspects of semantic similarity
- Legal text has domain-specific vocabulary — some models handle it better
- Dimension size affects storage and retrieval speed

How to use:
    python phase3_embeddings.py

    # To embed only a specific chunk size:
    python phase3_embeddings.py --chunk-size 512
"""

import os
import json
import argparse
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Set
from tqdm import tqdm
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer

load_dotenv()

CHUNKS_DIR     = Path(os.getenv("CHUNKS_DIR", "data/chunks"))
EMBEDDINGS_DIR = Path(os.getenv("EMBEDDINGS_DIR", "data/embeddings"))
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODELS = {
    "minilm": {
        "name":       "sentence-transformers/all-MiniLM-L6-v2",
        "short_name": "minilm",
        "dimensions": 384,
        "description": "Fast, lightweight. Good baseline."
    },
    "bge": {
        "name":       "BAAI/bge-base-en-v1.5",
        "short_name": "bge",
        "dimensions": 768,
        "description": "Higher quality semantic embeddings. Slower but better for legal text."
    }
}

BATCH_SIZE = 32  # Process this many chunks at once (adjust if GPU memory is tight)


# ─── Embedding Engine ─────────────────────────────────────────────────────────

class EmbeddingEngine:
    def __init__(self, model_key: str):
        config = EMBEDDING_MODELS[model_key]
        self.model_key   = model_key
        self.model_name  = config["name"]
        self.dimensions  = config["dimensions"]
        self.description = config["description"]

        print(f"\n  Loading embedding model: {self.model_name}")
        print(f"  Dimensions: {self.dimensions} | {self.description}")
        self.model = SentenceTransformer(self.model_name)
        print(f"  Model loaded.")

    def embed_chunks(self, chunks: List[dict]) -> np.ndarray:
        """
        Takes a list of chunk dicts, returns a numpy array of embeddings.
        Shape: (num_chunks, embedding_dim)
        """
        texts = [c["text"] for c in chunks]
        print(f"    Embedding {len(texts)} chunks in batches of {BATCH_SIZE}...")

        all_embeddings = []
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"    {self.model_key} batches"):
            batch = texts[i:i + BATCH_SIZE]
            batch_embeddings = self.model.encode(
                batch,
                normalize_embeddings=True,  # L2 normalize — important for cosine similarity
                show_progress_bar=False
            )
            all_embeddings.append(batch_embeddings)

        return np.vstack(all_embeddings).astype(np.float32)


# ─── Save / Load Helpers ─────────────────────────────────────────────────────

def save_embeddings(embeddings: np.ndarray, chunks: List[dict], model_key: str, chunk_size: int):
    """
    Save embeddings as .npy file and metadata as JSON.
    Naming: embeddings_<model>_<chunk_size>.npy
    """
    prefix = f"embeddings_{model_key}_{chunk_size}"

    # Save numpy array
    emb_path = EMBEDDINGS_DIR / f"{prefix}.npy"
    np.save(emb_path, embeddings)

    # Save metadata (chunks without the text to save space, just IDs and sources)
    meta = [{
        "chunk_id":    c["chunk_id"],
        "source":      c["source"],
        "chunk_index": c["chunk_index"],
        "char_length": c["char_length"],
        "text":        c["text"]  # keep text for later retrieval display
    } for c in chunks]

    meta_path = EMBEDDINGS_DIR / f"{prefix}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"    Saved: {emb_path.name} ({embeddings.shape}) + {meta_path.name}")
    return emb_path, meta_path


def load_embeddings(model_key: str, chunk_size: int):
    """Load saved embeddings and metadata."""
    prefix   = f"embeddings_{model_key}_{chunk_size}"
    emb_path = EMBEDDINGS_DIR / f"{prefix}.npy"
    meta_path = EMBEDDINGS_DIR / f"{prefix}_meta.json"

    if not emb_path.exists():
        raise FileNotFoundError(f"Embeddings not found: {emb_path}")

    embeddings = np.load(emb_path)
    with open(meta_path, 'r') as f:
        metadata = json.load(f)

    return embeddings, metadata


def extract_chunk_size(chunk_file: Path) -> Optional[int]:
    """
    Extract numeric chunk size from filenames like:
    <doc_name>_chunks_256.json
    """
    parts = chunk_file.stem.split("_chunks_")
    if len(parts) < 2:
        return None
    try:
        return int(parts[1])
    except ValueError:
        return None


def group_chunk_files_by_size(chunk_files: List[Path]) -> Dict[int, List[Path]]:
    """Group chunk JSON files by chunk size."""
    grouped: Dict[int, List[Path]] = {}
    for chunk_file in sorted(chunk_files):
        chunk_size = extract_chunk_size(chunk_file)
        if chunk_size is None:
            continue
        grouped.setdefault(chunk_size, []).append(chunk_file)
    return grouped


def load_and_merge_chunks(chunk_files: List[Path]) -> List[dict]:
    """
    Merge chunks from multiple document files into one corpus-level list.
    De-duplicates by chunk_id to avoid accidental duplicates.
    """
    merged_chunks: List[dict] = []
    seen_chunk_ids: Set[str] = set()

    for chunk_file in chunk_files:
        with open(chunk_file, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            chunk_id = chunk.get("chunk_id")
            if chunk_id and chunk_id in seen_chunk_ids:
                continue
            if chunk_id:
                seen_chunk_ids.add(chunk_id)
            merged_chunks.append(chunk)

    return merged_chunks


# ─── Main Loop ────────────────────────────────────────────────────────────────

def run_embedding_generation(chunk_sizes: List[int] = None):
    """
    For each chunk size × each embedding model, generate and save embeddings.
    IMPORTANT: embeddings are generated at CORPUS LEVEL (all documents merged),
    so each (model, chunk_size) output represents the full knowledge base.
    """
    available_chunk_files = list(CHUNKS_DIR.glob("*_chunks_*.json"))
    # Filter out paragraph and stats files
    chunk_files = [
        f for f in available_chunk_files
        if not f.name.endswith("_stats.json") and "paragraph" not in f.name
    ]

    if not chunk_files:
        print(f"\n[!] No chunk files found in {CHUNKS_DIR}/")
        print("    Run phase2_chunking.py first.\n")
        return

    grouped_files = group_chunk_files_by_size(chunk_files)

    # Filter by requested chunk sizes (if provided)
    if chunk_sizes:
        requested = set(chunk_sizes)
        grouped_files = {s: files for s, files in grouped_files.items() if s in requested}

    if not grouped_files:
        print("\n[!] No valid chunk files found for the requested chunk size(s).")
        return

    total_chunk_files = sum(len(files) for files in grouped_files.values())
    print(f"\n[+] Found {total_chunk_files} chunk file(s) across {len(grouped_files)} chunk size(s)")
    print(f"[+] Using {len(EMBEDDING_MODELS)} embedding models\n")

    stats = []

    for model_key in EMBEDDING_MODELS:
        print(f"\n{'─'*60}")
        print(f"  Embedding Model: {model_key.upper()} — {EMBEDDING_MODELS[model_key]['name']}")
        print(f"{'─'*60}")

        engine = EmbeddingEngine(model_key)

        for chunk_size in sorted(grouped_files.keys()):
            size_files = grouped_files[chunk_size]
            print(f"\n  Processing chunk_size={chunk_size} from {len(size_files)} file(s)")

            chunks = load_and_merge_chunks(size_files)

            if not chunks:
                print(f"  [!] Empty chunk file, skipping.")
                continue

            source_docs = len({c.get("source", "unknown") for c in chunks})
            print(f"    Merged chunks: {len(chunks)} from {source_docs} source document(s)")

            # Generate embeddings
            import time
            start_time = time.time()
            embeddings = engine.embed_chunks(chunks)
            elapsed = time.time() - start_time

            # Save
            save_embeddings(embeddings, chunks, model_key, chunk_size)

            stat = {
                "model":       model_key,
                "model_name":  EMBEDDING_MODELS[model_key]["name"],
                "chunk_size":  chunk_size,
                "num_chunk_files": len(size_files),
                "num_documents": source_docs,
                "num_chunks":  len(chunks),
                "emb_shape":   list(embeddings.shape),
                "dimensions":  EMBEDDING_MODELS[model_key]["dimensions"],
                "time_sec":    round(elapsed, 2),
                "files":       [f.name for f in size_files]
            }
            stats.append(stat)
            print(f"    Done in {elapsed:.1f}s | Shape: {embeddings.shape}")

    # Save stats
    stats_path = EMBEDDINGS_DIR / "embedding_stats.json"
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)

    print("\n─── Embedding Summary ────────────────────────────────────")
    print(f"  {'Model':<10} {'Chunk':>6} {'Chunks':>8} {'Dims':>6} {'Time(s)':>8}")
    print("  " + "-" * 44)
    for s in stats:
        print(f"  {s['model']:<10} {s['chunk_size']:>6} {s['num_chunks']:>8} {s['dimensions']:>6} {s['time_sec']:>8}")

    print(f"\n  Stats saved to {stats_path}")
    print("\n[✓] Phase 3 complete. Run phase4_vectordb.py next.")





In [14]:
run_embedding_generation()



[+] Found 30 chunk file(s) across 3 chunk size(s)
[+] Using 2 embedding models


────────────────────────────────────────────────────────────
  Embedding Model: MINILM — sentence-transformers/all-MiniLM-L6-v2
────────────────────────────────────────────────────────────

  Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
  Dimensions: 384 | Fast, lightweight. Good baseline.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

    Saved: embeddings_bge_256.npy ((619, 768)) + embeddings_bge_256_meta.json
    Done in 983.9s | Shape: (619, 768)

  Processing chunk_size=512 from 10 file(s)
    Merged chunks: 276 from 10 source document(s)
    Embedding 276 chunks in batches of 32...


    bge batches: 100%|██████████| 9/9 [12:12<00:00, 81.37s/it]


    Saved: embeddings_bge_512.npy ((276, 768)) + embeddings_bge_512_meta.json
    Done in 732.4s | Shape: (276, 768)

  Processing chunk_size=1024 from 10 file(s)
    Merged chunks: 136 from 10 source document(s)
    Embedding 136 chunks in batches of 32...


    bge batches: 100%|██████████| 5/5 [05:58<00:00, 71.70s/it]

    Saved: embeddings_bge_1024.npy ((136, 768)) + embeddings_bge_1024_meta.json
    Done in 358.5s | Shape: (136, 768)

─── Embedding Summary ────────────────────────────────────
  Model       Chunk   Chunks   Dims  Time(s)
  --------------------------------------------
  minilm        256      619    384   148.44
  minilm        512      276    384    58.65
  minilm       1024      136    384    28.93
  bge           256      619    768   983.87
  bge           512      276    768   732.37
  bge          1024      136    768   358.49

  Stats saved to data/embeddings/embedding_stats.json

[✓] Phase 3 complete. Run phase4_vectordb.py next.


In [15]:
"""
PHASE 4 — Vector Database Storage & Retrieval (Healthcare-ready)
================================================================
What this does:
- Loads embeddings generated in Phase 3
- Stores them in TWO vector databases: FAISS and ChromaDB
- Implements similarity search (retrieval) for both
- Lets you compare retrieval results between the two DBs
"""

import os
import json
import argparse
import numpy as np
import faiss
import chromadb
from pathlib import Path
from typing import List, Tuple
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

EMBEDDINGS_DIR = Path(os.getenv("EMBEDDINGS_DIR", "data/embeddings"))
VECTORDB_DIR = Path(os.getenv("VECTORDB_DIR", "vectordb"))
VECTORDB_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = int(os.getenv("TOP_K_RESULTS", 5))

EMBEDDING_MODELS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "bge": "BAAI/bge-base-en-v1.5",
}


class FAISSVectorDB:
    """
    FAISS-based vector store.
    Uses IndexFlatIP (inner product); embeddings are normalized,
    so IP is equivalent to cosine similarity.
    """
    def __init__(self, model_key: str, chunk_size: int):
        self.model_key = model_key
        self.chunk_size = chunk_size
        self.index = None
        self.metadata = []
        self.db_name = f"faiss_{model_key}_{chunk_size}"
        self.index_path = VECTORDB_DIR / f"{self.db_name}.index"
        self.meta_path = VECTORDB_DIR / f"{self.db_name}_meta.json"

    def build(self, embeddings: np.ndarray, metadata: List[dict]):
        print(f"\n  [FAISS] Building index: {self.db_name}")
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings.astype(np.float32))
        self.metadata = metadata

        faiss.write_index(self.index, str(self.index_path))
        with open(self.meta_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)

        print(f"  [FAISS] Index built: {self.index.ntotal} vectors, dim={dim}")
        print(f"  [FAISS] Saved to {self.index_path}")

    def load(self):
        if not self.index_path.exists():
            raise FileNotFoundError(f"FAISS index not found: {self.index_path}")
        if not self.meta_path.exists():
            raise FileNotFoundError(f"FAISS metadata not found: {self.meta_path}")

        self.index = faiss.read_index(str(self.index_path))
        with open(self.meta_path, "r", encoding="utf-8") as f:
            self.metadata = json.load(f)

        print(f"  [FAISS] Loaded index: {self.index.ntotal} vectors")

    def search(self, query_embedding: np.ndarray, top_k: int = TOP_K) -> List[dict]:
        if self.index is None:
            self.load()

        query = query_embedding.reshape(1, -1).astype(np.float32)
        scores, indices = self.index.search(query, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            item = self.metadata[idx].copy()
            item["score"] = float(score)
            item["db"] = "faiss"
            results.append(item)
        return results


class ChromaVectorDB:
    """
    ChromaDB-based vector store (persistent).
    """
    def __init__(self, model_key: str, chunk_size: int):
        self.model_key = model_key
        self.chunk_size = chunk_size
        self.collection_name = f"health_{model_key}_{chunk_size}"
        self.db_path = str(VECTORDB_DIR / "chromadb")
        self.client = chromadb.PersistentClient(path=self.db_path)
        self.collection = None

    def build(self, embeddings: np.ndarray, metadata: List[dict]):
        print(f"\n  [Chroma] Building collection: {self.collection_name}")

        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            pass

        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"},
        )

        ids = [m["chunk_id"] for m in metadata]
        docs = [m["text"] for m in metadata]
        metas = [{"source": m["source"], "chunk_index": m["chunk_index"]} for m in metadata]
        emb_list = embeddings.astype(np.float32).tolist()

        batch_size = 100
        for i in range(0, len(ids), batch_size):
            self.collection.add(
                ids=ids[i:i + batch_size],
                embeddings=emb_list[i:i + batch_size],
                documents=docs[i:i + batch_size],
                metadatas=metas[i:i + batch_size],
            )

        print(f"  [Chroma] Collection built: {self.collection.count()} vectors")
        print(f"  [Chroma] Persisted to {self.db_path}")

    def load(self):
        self.collection = self.client.get_collection(self.collection_name)
        print(f"  [Chroma] Loaded collection: {self.collection.count()} vectors")

    def search(self, query_embedding: np.ndarray, top_k: int = TOP_K) -> List[dict]:
        if self.collection is None:
            self.load()

        out = self.collection.query(
            query_embeddings=[query_embedding.astype(np.float32).tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )

        results = []
        for i, (doc, meta, dist) in enumerate(
            zip(out["documents"][0], out["metadatas"][0], out["distances"][0])
        ):
            results.append({
                "chunk_id": out["ids"][0][i],
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "score": round(1 - dist, 4),
                "db": "chromadb",
            })
        return results


class QueryEngine:
    """
    Query both FAISS and ChromaDB for side-by-side comparison.
    """
    def __init__(self, model_key: str, chunk_size: int):
        self.model_key = model_key
        self.chunk_size = chunk_size

        print(f"\n  Loading embedding model for queries: {EMBEDDING_MODELS[model_key]}")
        self.embedder = SentenceTransformer(EMBEDDING_MODELS[model_key])

        self.faiss_db = FAISSVectorDB(model_key, chunk_size)
        self.chroma_db = ChromaVectorDB(model_key, chunk_size)

        self.faiss_db.load()
        self.chroma_db.load()

    def embed_query(self, query: str) -> np.ndarray:
        return self.embedder.encode(query, normalize_embeddings=True)

    def search_both(self, query: str, top_k: int = TOP_K) -> Tuple[List[dict], List[dict]]:
        query_emb = self.embed_query(query)
        faiss_results = self.faiss_db.search(query_emb, top_k)
        chroma_results = self.chroma_db.search(query_emb, top_k)
        return faiss_results, chroma_results

    def compare_and_display(self, query: str, top_k: int = TOP_K):
        print(f"\n{'=' * 70}")
        print(f"  Query: {query}")
        print(f"{'=' * 70}")

        faiss_results, chroma_results = self.search_both(query, top_k)

        for rank, (f_res, c_res) in enumerate(zip(faiss_results, chroma_results), start=1):
            print(f"\n  Rank {rank}")
            print(f"  {'-' * 66}")
            print(f"  FAISS   score={f_res['score']:.4f} | source={f_res['source']}")
            print(f"  {f_res['text'][:200]}...")
            print(f"  Chroma  score={c_res['score']:.4f} | source={c_res['source']}")
            print(f"  {c_res['text'][:200]}...")

        faiss_ids = {r["chunk_id"] for r in faiss_results}
        chroma_ids = {r["chunk_id"] for r in chroma_results}
        overlap = len(faiss_ids & chroma_ids)
        print(f"\n  Overlap between FAISS and Chroma top-{top_k}: {overlap}/{top_k}")

        return faiss_results, chroma_results


def load_embeddings_for_config(model_key: str, chunk_size: int):
    emb_file = EMBEDDINGS_DIR / f"embeddings_{model_key}_{chunk_size}.npy"
    meta_file = EMBEDDINGS_DIR / f"embeddings_{model_key}_{chunk_size}_meta.json"

    if not emb_file.exists():
        raise FileNotFoundError(f"Embedding file not found: {emb_file}")
    if not meta_file.exists():
        raise FileNotFoundError(f"Metadata file not found: {meta_file}")

    embeddings = np.load(emb_file).astype(np.float32)
    with open(meta_file, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return embeddings, metadata


def build_single_vectordb(model_key: str, chunk_size: int):
    print(f"\n[+] Building vector DBs for model={model_key}, chunk_size={chunk_size}")
    embeddings, metadata = load_embeddings_for_config(model_key, chunk_size)

    faiss_db = FAISSVectorDB(model_key, chunk_size)
    faiss_db.build(embeddings, metadata)

    chroma_db = ChromaVectorDB(model_key, chunk_size)
    chroma_db.build(embeddings, metadata)

    print("\n[✓] Specific config build complete.")


def build_all_vectordbs():
    emb_files = sorted(EMBEDDINGS_DIR.glob("embeddings_*.npy"))

    if not emb_files:
        print(f"\n[!] No embedding files found in {EMBEDDINGS_DIR}/")
        print("    Run phase3 first.\n")
        return

    print(f"\n[+] Building vector DBs for {len(emb_files)} embedding file(s)\n")

    for emb_file in emb_files:
        parts = emb_file.stem.split("_")
        if len(parts) < 3:
            continue

        model_key = parts[1]
        if model_key not in EMBEDDING_MODELS:
            continue

        try:
            chunk_size = int(parts[2])
        except ValueError:
            continue

        print(f"\n  Config: model={model_key}, chunk_size={chunk_size}")
        embeddings, metadata = load_embeddings_for_config(model_key, chunk_size)

        faiss_db = FAISSVectorDB(model_key, chunk_size)
        faiss_db.build(embeddings, metadata)

        chroma_db = ChromaVectorDB(model_key, chunk_size)
        chroma_db.build(embeddings, metadata)

    print("\n[✓] Phase 4 complete. Run phase5 next.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Phase 4: build/query FAISS + ChromaDB")
    parser.add_argument("--model", type=str, choices=["minilm", "bge"], help="Embedding model key")
    parser.add_argument("--chunk-size", type=int, choices=[256, 512, 1024], help="Chunk size")
    parser.add_argument("--all", action="store_true", help="Build all available configs")
    parser.add_argument("--query", type=str, help="Query text for DB comparison")
    parser.add_argument("--top-k", type=int, default=TOP_K, help="Top-k results")
    args, _ = parser.parse_known_args()  # notebook-safe

    print("=" * 60)
    print("  PHASE 4 — Vector DB Storage & Retrieval (FAISS + ChromaDB)")
    print("=" * 60)

    if args.query:
        if not args.model or not args.chunk_size:
            parser.error("For --query, pass both --model and --chunk-size.")
        engine = QueryEngine(args.model, args.chunk_size)
        engine.compare_and_display(args.query, top_k=args.top_k)

    elif args.all:
        build_all_vectordbs()

    elif args.model and args.chunk_size:
        build_single_vectordb(args.model, args.chunk_size)

    else:
        parser.print_help()


  PHASE 4 — Vector DB Storage & Retrieval (FAISS + ChromaDB)
usage: colab_kernel_launcher.py [-h] [--model {minilm,bge}]
                                [--chunk-size {256,512,1024}] [--all]
                                [--query QUERY] [--top-k TOP_K]

Phase 4: build/query FAISS + ChromaDB

options:
  -h, --help            show this help message and exit
  --model {minilm,bge}  Embedding model key
  --chunk-size {256,512,1024}
                        Chunk size
  --all                 Build all available configs
  --query QUERY         Query text for DB comparison
  --top-k TOP_K         Top-k results


In [16]:
build_all_vectordbs()



[+] Building vector DBs for 6 embedding file(s)


  Config: model=bge, chunk_size=1024

  [FAISS] Building index: faiss_bge_1024
  [FAISS] Index built: 136 vectors, dim=768
  [FAISS] Saved to vectordb/faiss_bge_1024.index


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



  [Chroma] Building collection: health_bge_1024


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  [Chroma] Collection built: 136 vectors
  [Chroma] Persisted to vectordb/chromadb

  Config: model=bge, chunk_size=256

  [FAISS] Building index: faiss_bge_256
  [FAISS] Index built: 619 vectors, dim=768
  [FAISS] Saved to vectordb/faiss_bge_256.index

  [Chroma] Building collection: health_bge_256


...

  [Chroma] Collection built: 136 vectors
  [Chroma] Persisted to vectordb/chromadb

  Config: model=minilm, chunk_size=256

  [FAISS] Building index: faiss_minilm_256
  [FAISS] Index built: 619 vectors, dim=384
  [FAISS] Saved to vectordb/faiss_minilm_256.index

  [Chroma] Building collection: health_minilm_256


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  [Chroma] Collection built: 619 vectors
  [Chroma] Persisted to vectordb/chromadb

  Config: model=minilm, chunk_size=512

  [FAISS] Building index: faiss_minilm_512
  [FAISS] Index built: 276 vectors, dim=384
  [FAISS] Saved to vectordb/faiss_minilm_512.index

  [Chroma] Building collection: health_minilm_512


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  [Chroma] Collection built: 276 vectors
  [Chroma] Persisted to vectordb/chromadb

[✓] Phase 4 complete. Run phase5 next.


In [17]:
import os
import json
import argparse
import numpy as np
from pathlib import Path
from typing import List
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

OUTPUTS_DIR = Path(os.getenv("OUTPUTS_DIR", "outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = int(os.getenv("TOP_K_RESULTS", 5))

EMBEDDING_MODELS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "bge":    "BAAI/bge-base-en-v1.5"
}


class Retriever:
    def __init__(self, model_key: str, chunk_size: int, db_type: str = "faiss"):
        # Script mode import, notebook mode fallback to globals
        try:
            from phase4_vectordb import FAISSVectorDB, ChromaVectorDB
        except ImportError:
            FAISSVectorDB = globals().get("FAISSVectorDB")
            ChromaVectorDB = globals().get("ChromaVectorDB")
            if FAISSVectorDB is None or ChromaVectorDB is None:
                raise ImportError("FAISSVectorDB/ChromaVectorDB not found. Run Phase 4 code first.")

        print(f"  Loading embedder: {EMBEDDING_MODELS[model_key]}")
        self.embedder = SentenceTransformer(EMBEDDING_MODELS[model_key])

        if db_type == "faiss":
            self.db = FAISSVectorDB(model_key, chunk_size)
        elif db_type == "chromadb":
            self.db = ChromaVectorDB(model_key, chunk_size)
        else:
            raise ValueError("db_type must be 'faiss' or 'chromadb'")

        self.db.load()

    def retrieve(self, query: str, top_k: int = TOP_K) -> List[dict]:
        query_emb = self.embedder.encode(query, normalize_embeddings=True)
        return self.db.search(query_emb, top_k)


class CrossEncoderReranker:
    def __init__(self):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def rerank(self, query: str, chunks: List[dict]) -> List[dict]:
        if not chunks:
            return chunks
        pairs = [(query, c["text"]) for c in chunks]
        scores = self.model.predict(pairs)
        for chunk, score in zip(chunks, scores):
            chunk["rerank_score"] = float(score)
        return sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)


def build_healthcare_prompt(query: str, chunks: List[dict]) -> str:
    context_parts = []
    for i, chunk in enumerate(chunks):
        source = chunk.get("source", "unknown")
        text = chunk.get("text", "").strip()
        context_parts.append(f"[Source {i+1}: {source}]\n{text}")

    context = "\n\n".join(context_parts)

    prompt = f"""You are a healthcare document assistant.
Answer ONLY from the provided context.
If the answer is not in context, say exactly:
"I cannot find this information in the provided documents."
Keep answer factual, concise, and cite [Source X] lines you used.

=== CONTEXT ===
{context}
=== END CONTEXT ===

Question: {query}

Answer:"""
    return prompt


class LLMClient:
    def __init__(self, llm_identifier: str):
        parts = llm_identifier.split("/", 1)
        self.provider = parts[0]
        self.model_name = parts[1] if len(parts) > 1 else "llama3-8b-8192"

    def generate(self, prompt: str, max_tokens: int = 1024, temperature: float = 0.1) -> str:
        if self.provider == "groq":
            from groq import Groq
            client = Groq(api_key=os.getenv("GROQ_API_KEY"))
            response = client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content.strip()

        if self.provider == "together":
            from together import Together
            client = Together(api_key=os.getenv("TOGETHER_API_KEY"))
            response = client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content.strip()

        if self.provider == "ollama":
            import ollama
            response = ollama.chat(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                options={"num_predict": max_tokens, "temperature": temperature}
            )
            return response["message"]["content"].strip()

        raise ValueError("provider must be groq/together/ollama")


class HealthcareRAGPipeline:
    def __init__(
        self,
        model_key: str = "minilm",
        chunk_size: int = 512,
        db_type: str = "faiss",
        llm: str = "groq/llama3-8b-8192",
        use_reranker: bool = False,
        top_k: int = TOP_K
    ):
        self.config = {
            "embedding_model": model_key,
            "chunk_size": chunk_size,
            "vector_db": db_type,
            "llm": llm,
            "use_reranker": use_reranker,
            "top_k": top_k
        }
        self.retriever = Retriever(model_key, chunk_size, db_type)
        self.llm = LLMClient(llm)
        self.reranker = CrossEncoderReranker() if use_reranker else None

    def run(self, query: str, verbose: bool = True) -> dict:
        import time

        t0 = time.time()
        chunks = self.retriever.retrieve(query, self.config["top_k"])
        retrieval_time = time.time() - t0

        if self.reranker:
            chunks = self.reranker.rerank(query, chunks)

        prompt = build_healthcare_prompt(query, chunks)

        t1 = time.time()
        answer = self.llm.generate(prompt)
        generation_time = time.time() - t1

        result = {
            "query": query,
            "config": self.config,
            "retrieved_chunks": [
                {
                    "chunk_id": c.get("chunk_id"),
                    "source": c.get("source"),
                    "score": c.get("score", 0),
                    "text": c.get("text", "")[:300]
                } for c in chunks
            ],
            "prompt_length": len(prompt),
            "answer": answer,
            "retrieval_time": round(retrieval_time, 3),
            "generation_time": round(generation_time, 3),
            "total_time": round(retrieval_time + generation_time, 3)
        }

        if verbose:
            print("\nQuery:", query)
            print("Answer:", answer)

        return result

    def save_result(self, result: dict, filename: str = None):
        if filename is None:
            import datetime
            ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"result_{ts}.json"
        path = OUTPUTS_DIR / filename
        with open(path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        return path


# Backward compatibility if Phase 6 still refers to LegalRAGPipeline
LegalRAGPipeline = HealthcareRAGPipeline


In [21]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])
models = sorted([m.id for m in client.models.list().data])
print("\n".join(models))


allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
groq/compound
groq/compound-mini
llama-3.1-8b-instant
llama-3.3-70b-versatile
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
moonshotai/kimi-k2-instruct
moonshotai/kimi-k2-instruct-0905
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
whisper-large-v3
whisper-large-v3-turbo


In [22]:
ACTIVE_MODEL = "llama-3.1-8b-instant"

pipeline = HealthcareRAGPipeline(
    model_key="minilm",
    chunk_size=512,
    db_type="faiss",
    llm=f"groq/{ACTIVE_MODEL}",
    use_reranker=False
)

result = pipeline.run("What is vaccination and how does it work?", verbose=True)
pipeline.save_result(result)


  Loading embedder: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  [FAISS] Loaded index: 276 vectors

Query: What is vaccination and how does it work?
Answer: Vaccination is the process of introducing a vaccine into the body to stimulate the immune system to produce antibodies and memory cells that can recognize and fight specific germs. 

When a vaccine is introduced into the body, the immune system interacts with the germs present in the vaccine and produces proteins known as antibodies, which act against the germs in the vaccine. Certain cells also get programmed to remember this exposure to the germs, and when the germ enters the body again, these specialized cells known as memory cells get activated and produce very large quantities of antibodies, which surround the germs and kill them, thus preventing these germs from causing disease in the vaccinated person.

[Source 1: Immunization] 
[Source 4: Immunization]


PosixPath('outputs/result_20260331_165949.json')

In [23]:
import os
import json
import time
import pandas as pd
from pathlib import Path
from typing import List, Dict
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

OUTPUTS_DIR = Path(os.getenv("OUTPUTS_DIR", "outputs"))
LOGS_DIR = Path(os.getenv("LOGS_DIR", "logs"))
LOGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_LLM_1 = os.getenv("LLM_1", "groq/llama-3.1-8b-instant")
DEFAULT_LLM_2 = os.getenv("LLM_2", "groq/llama-3.1-8b-instant")

IN_DOMAIN_QUESTIONS = [
    "What is the difference between hereditary and congenital diseases?",
    "What are point mutations and how do they affect proteins?",
    "What is a frameshift mutation?",
    "How are X-ray images formed in diagnostic radiology?",
    "What are different modalities of X-ray imaging mentioned in the document?",
    "What is the difference between radiography and CT imaging?",
    "What is nutrition and how is it related to health?",
    "What are the main functions of food in the human body?",
    "Why is a balanced diet necessary?",
    "What is the definition of public health according to Winslow?",
    "What are the core functions of public health?",
    "What factors influence population health?",
    "What is vaccination and how does it work?",
    "Why are vaccines important for children?",
    "What diseases can be prevented through immunization?",
    "What are the key factors affecting mental health?",
    "What role do schools play in mental health wellbeing?",
    "What challenges are faced during adolescence?",
    "What is the purpose of the Infectious Diseases conference mentioned in the document?",
    "What are the main themes discussed in Infection 2022?",
    "What is cancer and how does it develop?",
    "What is metastasis?",
    "What is the difference between benign and malignant tumors?",
    "What is diabetes and what causes high blood glucose levels?",
    "What are the two major types of diabetes?",
    "How does insulin function in the body?",
    "What are the different types of cardiovascular diseases?",
    "What are the major risk factors of cardiovascular disease?",
    "What is the global impact of cardiovascular disease?"
]

TRAP_QUESTIONS = [
    "What are the latest AI techniques used in cancer diagnosis in 2024?",
    "What is the exact cure for diabetes mentioned in the document?",
    "What are WHO guidelines for COVID-19 vaccination in 2023?",
    "Which country has the highest cardiovascular disease rate according to the document?"
]

TEST_CASES = (
    [{"question": q, "question_type": "in_domain"} for q in IN_DOMAIN_QUESTIONS] +
    [{"question": q, "question_type": "trap"} for q in TRAP_QUESTIONS]
)

CONFIG_A = {
    "name": "Config_A_baseline_health",
    "model_key": "minilm",
    "chunk_size": 512,
    "db_type": "faiss",
    "llm": DEFAULT_LLM_1,
    "use_reranker": False
}

CONFIG_B = {
    "name": "Config_B_improved_health",
    "model_key": "bge",
    "chunk_size": 512,
    "db_type": "chromadb",
    "llm": DEFAULT_LLM_2,
    "use_reranker": True
}

FULL_EXPERIMENT_GRID = [
    {"name": "minilm_256_faiss",  "model_key": "minilm", "chunk_size": 256,  "db_type": "faiss",    "llm": DEFAULT_LLM_1, "use_reranker": False},
    {"name": "minilm_512_faiss",  "model_key": "minilm", "chunk_size": 512,  "db_type": "faiss",    "llm": DEFAULT_LLM_1, "use_reranker": False},
    {"name": "minilm_1024_faiss", "model_key": "minilm", "chunk_size": 1024, "db_type": "faiss",    "llm": DEFAULT_LLM_1, "use_reranker": False},
    {"name": "bge_256_chroma",    "model_key": "bge",    "chunk_size": 256,  "db_type": "chromadb", "llm": DEFAULT_LLM_2, "use_reranker": True},
    {"name": "bge_512_chroma",    "model_key": "bge",    "chunk_size": 512,  "db_type": "chromadb", "llm": DEFAULT_LLM_2, "use_reranker": True},
    {"name": "bge_1024_chroma",   "model_key": "bge",    "chunk_size": 1024, "db_type": "chromadb", "llm": DEFAULT_LLM_2, "use_reranker": True},
]

def _resolve_pipeline_class():
    cls = globals().get("HealthcareRAGPipeline") or globals().get("LegalRAGPipeline")
    if cls is not None:
        return cls
    try:
        from phase5_rag_pipeline import HealthcareRAGPipeline as cls2
        return cls2
    except Exception:
        from phase5_rag_pipeline import LegalRAGPipeline as cls3
        return cls3

def auto_score_retrieval(chunks: List[dict]) -> Dict[str, float]:
    if not chunks:
        return {"avg_retrieval_score": 0, "max_retrieval_score": 0, "min_retrieval_score": 0}
    scores = [c.get("score", 0) for c in chunks]
    return {
        "avg_retrieval_score": round(sum(scores) / len(scores), 4),
        "max_retrieval_score": round(max(scores), 4),
        "min_retrieval_score": round(min(scores), 4),
    }

def is_safe_refusal(answer: str) -> bool:
    a = (answer or "").lower()
    markers = [
        "i cannot find this information in the provided documents",
        "not found in the provided documents",
        "not in the provided documents",
        "insufficient context",
        "cannot be determined from the provided context",
    ]
    return any(m in a for m in markers)

def run_config(config: dict, test_cases: List[dict]) -> List[dict]:
    PipelineClass = _resolve_pipeline_class()

    pipeline = PipelineClass(
        model_key=config["model_key"],
        chunk_size=config["chunk_size"],
        db_type=config["db_type"],
        llm=config["llm"],
        use_reranker=config.get("use_reranker", False),
    )

    results = []
    for i, case in enumerate(tqdm(test_cases, desc=config["name"])):
        q = case["question"]
        qtype = case["question_type"]

        try:
            result = pipeline.run(q, verbose=False)
            retrieval_scores = auto_score_retrieval(result["retrieved_chunks"])

            trap_safe_refusal = int(is_safe_refusal(result["answer"])) if qtype == "trap" else None
            trap_hallucination_flag = int(qtype == "trap" and not is_safe_refusal(result["answer"]))

            row = {
                "config_name": config["name"],
                "embedding_model": config["model_key"],
                "chunk_size": config["chunk_size"],
                "vector_db": config["db_type"],
                "llm": config["llm"],
                "question_id": i + 1,
                "question_type": qtype,
                "question": q,
                "answer": result["answer"],
                "answer_length": len(result["answer"]),
                "retrieval_time": result["retrieval_time"],
                "generation_time": result["generation_time"],
                "total_time": result["total_time"],
                "manual_score": -1,
                "trap_safe_refusal": trap_safe_refusal,
                "trap_hallucination_flag": trap_hallucination_flag,
                **retrieval_scores,
            }
            results.append(row)

        except Exception as e:
            results.append({
                "config_name": config["name"],
                "question_id": i + 1,
                "question_type": qtype,
                "question": q,
                "answer": f"ERROR: {str(e)}",
                "error": True,
            })

        time.sleep(0.4)

    return results

def save_results(results: List[dict], config_name: str):
    path = LOGS_DIR / f"results_{config_name}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Saved: {path}")
    return path

def generate_comparison_report():
    result_files = sorted(LOGS_DIR.glob("results_*.json"))
    if not result_files:
        print("[!] No result files found.")
        return

    all_rows = []
    for rf in result_files:
        with open(rf, "r", encoding="utf-8") as f:
            all_rows.extend(json.load(f))

    df = pd.DataFrame(all_rows)
    if "error" in df.columns:
        df = df[df["error"] != True].copy()

    numeric_cols = [
        "retrieval_time", "generation_time", "total_time",
        "avg_retrieval_score", "max_retrieval_score", "answer_length"
    ]
    available = [c for c in numeric_cols if c in df.columns]

    if "config_name" in df.columns and available:
        df.groupby("config_name")[available].mean().round(4).to_csv(OUTPUTS_DIR / "table1_config_comparison.csv")

    if "chunk_size" in df.columns and available:
        df.groupby("chunk_size")[available].mean().round(4).to_csv(OUTPUTS_DIR / "table2_chunk_comparison.csv")

    if "embedding_model" in df.columns and available:
        df.groupby("embedding_model")[available].mean().round(4).to_csv(OUTPUTS_DIR / "table3_embedding_comparison.csv")

    if "vector_db" in df.columns and available:
        df.groupby("vector_db")[available].mean().round(4).to_csv(OUTPUTS_DIR / "table4_vectordb_comparison.csv")

    answer_cols = ["config_name", "question_id", "question_type", "question", "answer", "manual_score"]
    answer_cols = [c for c in answer_cols if c in df.columns]
    df[answer_cols].to_csv(OUTPUTS_DIR / "table5_answers_for_scoring.csv", index=False)

    if "question_type" in df.columns and "trap_hallucination_flag" in df.columns:
        trap_df = df[df["question_type"] == "trap"].copy()
        if not trap_df.empty:
            trap_df.groupby("config_name")[["trap_safe_refusal", "trap_hallucination_flag"]].mean().round(4).to_csv(
                OUTPUTS_DIR / "table6_trap_hallucination_comparison.csv"
            )

    print("[✓] Report tables generated in outputs/")

def run_compare():
    for config in [CONFIG_A, CONFIG_B]:
        results = run_config(config, TEST_CASES)
        save_results(results, config["name"])
    generate_comparison_report()

def run_full_grid():
    for config in FULL_EXPERIMENT_GRID:
        results = run_config(config, TEST_CASES)
        save_results(results, config["name"])
    generate_comparison_report()


In [24]:
run_compare()


  Loading embedder: sentence-transformers/all-MiniLM-L6-v2
  [FAISS] Loaded index: 276 vectors


Config_A_baseline_health: 100%|██████████| 33/33 [14:07<00:00, 25.69s/it]


Saved: logs/results_Config_A_baseline_health.json
  Loading embedder: BAAI/bge-base-en-v1.5


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


  [Chroma] Loaded collection: 276 vectors


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Config_B_improved_health: 100%|██████████| 33/33 [14:42<00:00, 26.75s/it]

Saved: logs/results_Config_B_improved_health.json
[✓] Report tables generated in outputs/


In [25]:
run_full_grid()


  Loading embedder: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  [FAISS] Loaded index: 619 vectors


minilm_256_faiss: 100%|██████████| 33/33 [07:08<00:00, 12.98s/it]


Saved: logs/results_minilm_256_faiss.json
  Loading embedder: sentence-transformers/all-MiniLM-L6-v2
  [FAISS] Loaded index: 276 vectors


...

Saved: logs/results_bge_512_chroma.json
  Loading embedder: BAAI/bge-base-en-v1.5


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


  [Chroma] Loaded collection: 136 vectors


bge_1024_chroma: 100%|██████████| 33/33 [01:46<00:00,  3.23s/it]

Saved: logs/results_bge_1024_chroma.json
[✓] Report tables generated in outputs/


In [26]:
import os, shutil, glob
from pathlib import Path
from google.colab import files

EXPORT_DIR = Path("assignment_export")
ZIP_NAME = "assignment_export.zip"

# fresh export folder
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# copy runtime-generated folders (preserve structure)
for d in ["data", "vectordb", "logs", "outputs", "sample_data"]:
    src = Path(d)
    if src.exists():
        shutil.copytree(src, EXPORT_DIR / d)

# copy top-level useful files if present
for pattern in ["*.py", "*.ipynb", "requirements.txt", ".env", "*.md", "*.txt", "*.json", "*.csv"]:
    for f in glob.glob(pattern):
        p = Path(f)
        if p.is_file():
            shutil.copy2(p, EXPORT_DIR / p.name)

# zip
if Path(ZIP_NAME).exists():
    Path(ZIP_NAME).unlink()
shutil.make_archive("assignment_export", "zip", root_dir=".", base_dir=str(EXPORT_DIR))

# quick check
print("Created:", ZIP_NAME)
print("Size (MB):", round(Path(ZIP_NAME).stat().st_size / (1024*1024), 2))

# download
files.download(ZIP_NAME)


Created: assignment_export.zip
Size (MB): 69.6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>